# My Notes: Getting Started with Hugging Face Pipelines

I'm starting my journey into LLM engineering. The Hugging Face `transformers` library is my primary toolset for open-source AI.

**Pipelines** are my entry point. They handle the heavy lifting (tokenization, loading models, post-processing) automatically.

### My Workflow:
1. **Initialize**: `my_pipeline = pipeline("task-name")` (Creates the object)
2. **Run**: `result = my_pipeline("input")` (Gets the answer)

## Core Concept: Inference vs. Training

I need to distinguish between these two phases:

### 1. Training & Fine-Tuning
This is the 'learning' phase. **Training** is starting from zero. **Fine-tuning** is taking a model that already knows things (pre-trained) and giving it specific knowledge from a smaller dataset.

### 2. Inference
This is the 'using' phase. When I run a model to get a prediction or generation, I'm doing inference. **The Pipelines API is my go-to for easy inference.**

In [ ]:
# Setting up my environment with the essential libraries
# - transformers: for the core models
# - datasets: to grab training/testing data
# - accelerate: to make things run faster on GPU
!pip install -q --upgrade datasets transformers diffusers accelerate

In [ ]:
# Always check for GPU availability first.
# Using the Colab T4 (CUDA) is essential for faster model processing.
import torch
if torch.cuda.is_available():
    print(f"GPU ready: {torch.cuda.get_device_name(0)}")
else:
    print("Note: Running on CPU, which will be much slower.")

In [ ]:
# Imports
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline
from diffusers import DiffusionPipeline
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio

In [ ]:
hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("HF key looks good so far")
else:
  print("HF key is not set - please click the key in the left sidebar")
login(hf_token, add_to_git_credential=True)

## My Guide to Using Pipelines

Pipelines help me run inference for common tasks without manually setting up tokenizers.

**Step 1: Setup**
Specify the `task`, the `model` (optional, but good for control), and the `device` (use 'cuda' for GPU).

**Step 2: Execution**
Pass my data into the pipeline object.

In [ ]:
# Task 1: Sentiment Analysis
# I'll default to 'cuda' if I have a GPU to speed this up.
classifier = pipeline("sentiment-analysis", device="cuda" if torch.cuda.is_available() else "cpu")

# Run the classifier on a sample string
result = classifier("I am learning how to use Hugging Face and it's amazing!")
print(f"Result: {result}")

In [ ]:
result = my_simple_sentiment_analyzer("I should be more excited to be on the way to LLM mastery!")
print(result)

In [ ]:
better_sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device="cuda")
result = better_sentiment("I should be more excited to be on the way to LLM mastery!!")
print(result)

In [ ]:
# Example 2: Named Entity Recognition (NER)
# This identifies real-world objects like people, places, and organizations.
ner_pipe = pipeline("ner", device="cuda" if torch.cuda.is_available() else "cpu")
text = "Hugging Face is based in New York and was founded by Clement Delangue."
for entity in ner_pipe(text):
    print(entity)

In [ ]:
# Example 3: Question Answering
# Provide a 'context' (knowledge) and ask a question based on it.
qa_pipe = pipeline("question-answering", device="cuda" if torch.cuda.is_available() else "cpu")
context = "Pipelines are the simplest way to use models for inference."
question = "What are pipelines used for?"

result = qa_pipe(question=question, context=context)
print(f"Answer: {result['answer']}")

In [ ]:
# Text Summarization
# I can control the length of the summary using max_length and min_length
summarizer = pipeline("summarization", device="cuda" if torch.cuda.is_available() else "cpu")

long_text = """
The Hugging Face transformers library is an incredibly versatile and powerful tool for natural language processing (NLP).
It allows users to perform a wide range of tasks such as text classification, named entity recognition, and question answering, among others.
It's an extremely popular library that's widely used by the open-source data science community.
It lowers the barrier to entry into the field by providing Data Scientists with a productive, convenient way to work with transformer models.
"""

summary = summarizer(long_text, max_length=50, min_length=25, do_sample=False)
print(f"Summary: {summary[0]['summary_text']}")

In [ ]:
# Example 4: Translation
# You can specify the direction of translation directly in the task name.
translator = pipeline("translation_en_to_fr", device="cuda" if torch.cuda.is_available() else "cpu")
result = translator("Learning AI is a journey, not a destination.")
print(result[0]['translation_text'])

In [ ]:
# Another translation, showing a model being specified
# All translation models are here: https://huggingface.co/models?pipeline_tag=translation&sort=trending
translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es", device="cuda")
result = translator("The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API.")
print(result[0]['translation_text'])

In [ ]:
# Classification
classifier = pipeline("zero-shot-classification", device="cuda")
result = classifier("Hugging Face's Transformers library is amazing!", candidate_labels=["technology", "sports", "politics"])
print(result)

In [ ]:
# Text Generation
generator = pipeline("text-generation", device="cuda")
result = generator("If there's one thing I want you to remember about using HuggingFace pipelines, it's")
print(result[0]['generated_text'])

In [ ]:
# Image Generation - remember this?! Now you know what's going on
# Pipelines can be used for diffusion models as well as transformers
from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
pipe.to("cuda")
prompt = "A class of students learning AI engineering in a vibrant pop-art style"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
display(image)

In [ ]:
# Audio Generation
from transformers import pipeline
from datasets import load_dataset
import soundfile as sf
import torch
from IPython.display import Audio

synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device='cuda')
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speech = synthesiser("Hi to an artificial intelligence engineer, on the way to mastery!", forward_params={"speaker_embeddings": speaker_embedding})

Audio(speech["audio"], rate=speech["sampling_rate"])

# My Learning Resources

To continue my journey, I should bookmark these:

- [Transformers Pipeline Docs](https://huggingface.co/docs/transformers/main_classes/pipelines)
- [The Model Hub](https://huggingface.co/models) - This is where I find specific models for my projects.